In [1]:
import os, glob, hashlib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATASET_ROOT = os.environ.get("DATASET_ROOT", "/home/jovyan/shared/donut/Alzheimer/OAS1")
BASE_SEED = 1334
OTHER_SEEDS = [0, 1, 42, 2024, 3407]
CHECK_ID = "0454"


In [2]:
df = pd.read_excel(f"{DATASET_ROOT}/_oasis_cross-sectional.xlsx")
df = df[df["CDR"].notna()]
df["label"] = (df["CDR"] > 0).astype(int)

def find_file(subject_id):
    pattern = f"{DATASET_ROOT}/{subject_id}/PROCESSED/MPRAGE/T88_111/*_masked_gfc.hdr"
    files = glob.glob(pattern)
    return files[0] if files else None

df["path"] = df["ID"].apply(find_file)
df = df[df["path"].notna()]
df["subject_id"] = df["ID"].apply(lambda x: x.split("_")[1])
print("df หลังกรอง:", df.shape, "| subject ไม่ซ้ำ:", df["subject_id"].nunique())


df หลังกรอง: (234, 15) | subject ไม่ซ้ำ: 234


In [3]:
def split_ids(df, seed):
    # ลอจิกเดียวกับโน้ตบุ๊กทั้ง 3 โมเดล คืนเป็นรายชื่อ subject ของแต่ละชุด
    sl = df.groupby("subject_id")["label"].first()
    ids, vals = list(sl.index), list(sl.values)
    tr, temp = train_test_split(ids, test_size=0.3, stratify=vals, random_state=seed)
    lab = dict(zip(ids, vals))
    va, te = train_test_split(temp, test_size=0.5, stratify=[lab[s] for s in temp], random_state=seed)
    return set(tr), set(va), set(te)

def where(split, sid):
    for name, s in zip(("train", "val", "test"), split):
        if sid in s:
            return name
    return "ไม่พบ"

def ratio(split_set):
    lab = df.groupby("subject_id")["label"].first()
    return lab.loc[list(split_set)].mean()


In [4]:
base = split_ids(df, BASE_SEED)
tr, va, te = base
print(f"=== ตรวจทีละคู่ (seed {BASE_SEED}) ===")
print(f"train={len(tr)} val={len(va)} test={len(te)} | รวม={len(tr|va|te)} / ทั้งหมด={df['subject_id'].nunique()}")
print(f"train ∩ val  = {len(tr & va)}")
print(f"train ∩ test = {len(tr & te)}")
print(f"val   ∩ test = {len(va & te)}")
print(f"ตัดกันทั้ง 3 ชุดพร้อมกัน (assert เดิม) = {len(tr & va & te)}  <- ว่างเสมอ ตรวจอะไรไม่ได้")
print(f"คนที่หลุดไม่อยู่ชุดไหน = {len(set(df['subject_id']) - (tr|va|te))}")
assert not (tr & va) and not (tr & te) and not (va & te)
print(f"สัดส่วนอัลไซเมอร์: train={ratio(tr):.3f} val={ratio(va):.3f} test={ratio(te):.3f}")
print(f"{CHECK_ID} -> {where(base, CHECK_ID)}")


=== ตรวจทีละคู่ (seed 1334) ===
train=163 val=35 test=36 | รวม=234 / ทั้งหมด=234
train ∩ val  = 0
train ∩ test = 0
val   ∩ test = 0
ตัดกันทั้ง 3 ชุดพร้อมกัน (assert เดิม) = 0  <- ว่างเสมอ ตรวจอะไรไม่ได้
คนที่หลุดไม่อยู่ชุดไหน = 0
สัดส่วนอัลไซเมอร์: train=0.429 val=0.429 test=0.417
0454 -> val


In [5]:
rows = []
for seed in [BASE_SEED] + OTHER_SEEDS:
    s = split_ids(df, seed)
    a, b, c = s
    own_overlap = len(a & b) + len(a & c) + len(b & c)
    # คนที่อยู่คนละชุดเมื่อเทียบกับ seed หลัก
    moved = sum(where(s, sid) != where(base, sid) for sid in df["subject_id"])
    rows.append({
        "seed": seed, "train": len(a), "val": len(b), "test": len(c),
        "overlap_ในตัวเอง": own_overlap,
        "คนย้ายชุดเทียบ_1334": moved,
        f"{CHECK_ID}_อยู่": where(s, CHECK_ID),
        "ratio_train/val/test": f"{ratio(a):.2f}/{ratio(b):.2f}/{ratio(c):.2f}",
    })
res = pd.DataFrame(rows)
print(res.to_string(index=False))


 seed  train  val  test  overlap_ในตัวเอง  คนย้ายชุดเทียบ_1334 0454_อยู่ ratio_train/val/test
 1334    163   35    36                 0                    0       val       0.43/0.43/0.42
    0    163   35    36                 0                  100     train       0.43/0.43/0.42
    1    163   35    36                 0                  115     train       0.43/0.43/0.42
   42    163   35    36                 0                  114     train       0.43/0.43/0.42
 2024    163   35    36                 0                  103     train       0.43/0.43/0.42
 3407    163   35    36                 0                  111      test       0.43/0.43/0.42


In [6]:
print(f"=== โมเดล A ใช้ seed {BASE_SEED} / โมเดล B ใช้ seed อื่น ===")
print("นับ: คนที่อยู่ TRAIN ของ A แต่ไปอยู่ VAL/TEST ของ B (leakage ข้ามโมเดล)\n")
a_train = base[0]
rows = []
for seed in OTHER_SEEDS:
    b = split_ids(df, seed)
    rows.append({
        "seed ของ B": seed,
        "train_A ไป val_B": len(a_train & b[1]),
        "train_A ไป test_B": len(a_train & b[2]),
        "รวมที่ A เทรนแล้ว B เอามาประเมิน": len(a_train & (b[1] | b[2])),
        "% ของ test_B": f"{100 * len(a_train & b[2]) / len(b[2]):.0f}%",
    })
print(pd.DataFrame(rows).to_string(index=False))
print()
print("เทียบ: ถ้า seed เดียวกัน ตัวเลขทั้งหมดนี้ต้องเป็น 0")
same = split_ids(df, BASE_SEED)
print("seed เดียวกัน train_A ∩ (val_B ∪ test_B) =", len(a_train & (same[1] | same[2])))


=== โมเดล A ใช้ seed 1334 / โมเดล B ใช้ seed อื่น ===
นับ: คนที่อยู่ TRAIN ของ A แต่ไปอยู่ VAL/TEST ของ B (leakage ข้ามโมเดล)

 seed ของ B  train_A ไป val_B  train_A ไป test_B  รวมที่ A เทรนแล้ว B เอามาประเมิน % ของ test_B
          0                21                 24                                45          67%
          1                24                 29                                53          81%
         42                27                 25                                52          69%
       2024                24                 21                                45          58%
       3407                26                 25                                51          69%

เทียบ: ถ้า seed เดียวกัน ตัวเลขทั้งหมดนี้ต้องเป็น 0
seed เดียวกัน train_A ∩ (val_B ∪ test_B) = 0


In [7]:
# ทดลอง: แต่ละโมเดลใช้ seed คนละค่า (แก้ค่าตรงนี้ได้)
MODEL_SEEDS = {"2D": 1334, "2.5D": 42, "3D": 2024}

splits = {m: split_ids(df, sd) for m, sd in MODEL_SEEDS.items()}

print("=== ภายในแต่ละโมเดล (ยังถูกต้องไหม) ===")
for m, (a, b, c) in splits.items():
    ok = not (a & b) and not (a & c) and not (b & c)
    print(f"{m:5s} seed={MODEL_SEEDS[m]:<5} train={len(a)} val={len(b)} test={len(c)} | "
          f"overlap ทีละคู่={len(a&b)}/{len(a&c)}/{len(b&c)} -> {'ผ่าน' if ok else 'ไม่ผ่าน'} | "
          f"ratio {ratio(a):.2f}/{ratio(b):.2f}/{ratio(c):.2f} | {CHECK_ID} -> {where((a, b, c), CHECK_ID)}")

print()
print("=== ข้ามโมเดล (ชุดเดียวกันไหม) ===")
names = list(splits)
pool = df["subject_id"]
conflict = [sid for sid in pool if len({where(splits[m], sid) for m in names}) > 1]
print(f"คนที่อยู่ชุดไม่ตรงกันอย่างน้อย 1 โมเดล: {len(conflict)} / {len(pool)} คน ({100*len(conflict)/len(pool):.0f}%)")

print()
print("leakage: คนที่อยู่ TRAIN ของโมเดลแถว แต่ไปเป็น VAL/TEST ของโมเดลคอลัมน์")
mat = pd.DataFrame(index=names, columns=names, dtype=object)
for x in names:
    for y in names:
        mat.loc[x, y] = len(splits[x][0] & (splits[y][1] | splits[y][2]))
print(mat.to_string())
print()
print("เทียบ: ทุกโมเดลใช้ seed เดียวกัน ค่าในตารางต้องเป็น 0 ทั้งหมด")


=== ภายในแต่ละโมเดล (ยังถูกต้องไหม) ===
2D    seed=1334  train=163 val=35 test=36 | overlap ทีละคู่=0/0/0 -> ผ่าน | ratio 0.43/0.43/0.42 | 0454 -> val
2.5D  seed=42    train=163 val=35 test=36 | overlap ทีละคู่=0/0/0 -> ผ่าน | ratio 0.43/0.43/0.42 | 0454 -> train
3D    seed=2024  train=163 val=35 test=36 | overlap ทีละคู่=0/0/0 -> ผ่าน | ratio 0.43/0.43/0.42 | 0454 -> train

=== ข้ามโมเดล (ชุดเดียวกันไหม) ===
คนที่อยู่ชุดไม่ตรงกันอย่างน้อย 1 โมเดล: 149 / 234 คน (64%)

leakage: คนที่อยู่ TRAIN ของโมเดลแถว แต่ไปเป็น VAL/TEST ของโมเดลคอลัมน์
      2D 2.5D  3D
2D     0   52  45
2.5D  52    0  46
3D    45   46   0

เทียบ: ทุกโมเดลใช้ seed เดียวกัน ค่าในตารางต้องเป็น 0 ทั้งหมด
